In [1]:
import os, gc, random, re
import torch
from torch.utils.data import DataLoader
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from tqdm import tqdm
import json
from fastchat.conversation import get_conv_template as get_conversation_template

/home/yang/.conda/envs/pat/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
file_path = "MMLU_data.json"

with open(file_path, 'r', encoding='utf-8') as f:
    combined_data = json.load(f)

updated_input = [item[0] for item in combined_data]
question = [item[1] for item in combined_data]
subject = [item[2] for item in combined_data]
choices = [item[3] for item in combined_data]
answer = [item[4] for item in combined_data]

In [3]:
model_name = "./PAT/models/Llama-2-7b-chat-hf"
quantization_config = BitsAndBytesConfig(load_in_8bit=True)
access_token = ""
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    trust_remote_code=True,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

device = next(model.parameters()).device

/home/yang/.conda/envs/pat/lib/python3.11/site-packages/transformers/quantizers/auto.py:222: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)
Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████████████████████████████| 2/2 [00:06<00:00,  3.22s/it]


In [4]:
def format_prompt(prompts):

    formatted_prompts = []
    answer = "The correct answer is"

    if "llama-3" in model_name.lower():
        template_name = "llama-3"
        json_path = "./PAT/results/llama3_gcg_defense.json"
    elif "vicuna" in model_name.lower():
        template_name = "vicuna_v1.1"
        json_path = "./PAT/results/vicuna13b_gcg_defense.json"
    elif "llama-2" in model_name.lower():
        template_name = "llama-2"
        json_path = "./PAT/results/llama2_gcg_defense.json"
    elif "deepseek" in model_name.lower():
        template_name = "deepseek-chat"
        json_path = "./PAT/results/deepseek_gcg_defense.json"
    elif "openchat" in model_name.lower():
        template_name = "openchat_3.5"
        json_path = "./PAT/results/openchat_gcg_defense.json"
    else:
        print(f"A chat template for '{model_name}' is not defined.")
        return None

    with open(json_path, "r") as f:
        defense_data = json.load(f)
        defense_prefix = defense_data["def_controls"][1]

    for prompt in prompts:
        conv_template = get_conversation_template(template_name)
        conv_template.append_message(conv_template.roles[0], f"{defense_prefix} {prompt}") # User role
        # conv_template.append_message(conv_template.roles[1], answer) # Assistant role
        conv_template.append_message(conv_template.roles[1], None) # Assistant role
        final_prompt = conv_template.get_prompt()
        formatted_prompts.append(final_prompt)
        
    return formatted_prompts

In [5]:
def generate_text(prompts, batch_size=128):
    generated_texts = []
    dataloader = DataLoader(prompts, batch_size=batch_size, shuffle=False)

    for batch in tqdm(dataloader, desc="Generating responses", total=len(prompts)//batch_size + 1):
        # Tokenize the batch
        tokenized_inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
        # Generate
        with torch.no_grad():
            generated_ids = model.generate(input_ids=tokenized_inputs["input_ids"],
                                           attention_mask=tokenized_inputs["attention_mask"],
                                           max_new_tokens=64, pad_token_id=tokenizer.pad_token_id,
                                           do_sample=False, repetition_penalty=1)

        # Decode
        for i in range(len(batch)):
            output_tokens = generated_ids[i][tokenized_inputs["input_ids"].size(1):].cpu()
            generated_text = tokenizer.decode(output_tokens, skip_special_tokens=True)
            generated_texts.append(generated_text)

        # Clear GPU memory
        del tokenized_inputs, generated_ids, output_tokens, generated_text
        torch.cuda.empty_cache()
        gc.collect()

    return generated_texts

In [6]:
results = []

format_clean_prompts = format_prompt(updated_input)

# Process clean prompts
clean_outputs = generate_text(format_clean_prompts)
for prompt, output in zip(updated_input, clean_outputs):
    results.append({"model": model_name, "input": prompt, "output": output})

# Clear GPU memory
del model, tokenizer, format_clean_prompts
torch.cuda.empty_cache()
gc.collect()

df_inference = pd.DataFrame(results)
df_inference["subject"] = subject
df_inference["answer"] = answer

# Save to CSV
df_inference.to_csv("evaluation_llama2_MMLU_PAT.csv", index=False)
print("Processing complete. Results saved to evaluation_llama2_MMLU_PAT.csv.")

Generating responses:   0%|                                                                                             | 0/11 [00:00<?, ?it/s]Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
/home/yang/.conda/envs/pat/lib/python3.11/site-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
Generating responses: 100%|████████████████████████████████████████████████████████████████████████████████████| 11/11 [02:55<00:00, 15.96s/it]

Processing complete. Results saved to evaluation_llama2_MMLU_PAT.csv.


In [7]:
import pandas as pd
df = pd.read_csv('evaluation_llama2_MMLU_PAT.csv')
pd.set_option('display.max_colwidth', None)

In [8]:
def extract_answer_index(text):
    match = re.search(r"(?:^|(?<=\W))\s*([A-D])\s*(?:$|(?=\W))", text)
    if match:
        letter = match.group(1)
        return {"A": 0, "B": 1, "C": 2, "D": 3}[letter]
    return -1

df["output"] = df["output"].apply(lambda x: x if isinstance(x, str) else "")
df["predicted_answer"] = df["output"].apply(extract_answer_index)

In [9]:
df["correct"] = df["predicted_answer"] == df["answer"]
accuracy = df["correct"].mean()

print(f"Accuracy on {len(df)} examples: {accuracy:.2%}")

Accuracy on 1404 examples: 38.11%
